In [ ]:
# ============================================================
# 0. 기본 세팅 (라이브러리 임포트, 시드, 디바이스)
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
import time

import torch
import torch.nn as nn
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score
)

warnings.filterwarnings("ignore")

# 재현성을 위한 시드
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# 디바이스 선택 (Colab T4 기준)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)


In [ ]:
# ============================================================
# 1. 데이터 로딩 및 기본 분석
# ============================================================

# Colab 기준: otc_train.csv 를 /content 에 업로드했다고 가정
DATA_PATH = "/content/otc_train.csv"  # 필요하면 경로 수정

df = pd.read_csv(DATA_PATH)
print("===== Raw Data Head =====")
print(df.head())
print("=========================")
print("컬럼:", df.columns.tolist())

n_interactions = len(df)
n_users_raw = df["user"].nunique()
n_items_raw = df["item"].nunique()

print(f"총 상호작용 수: {n_interactions}")
print(f"유저 수(user 컬럼 기준): {n_users_raw}")
print(f"아이템 수(item 컬럼 기준): {n_items_raw}")

print("\nrating 통계 (Bitcoin-OTC의 신뢰도 점수, 모델에는 직접 사용하지 않음):")
print(df["rating"].describe())

# ============================================================
# 2. ID 인덱싱 및 K (추천 개수) 규칙 정의
# ============================================================

# user / item 을 0 ~ n-1 인덱스로 매핑
user2idx = {u: i for i, u in enumerate(sorted(df["user"].unique()))}
item2idx = {v: i for i, v in enumerate(sorted(df["item"].unique()))}

idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: v for v, i in item2idx.items()}

df["user_idx"] = df["user"].map(user2idx)
df["item_idx"] = df["item"].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)
print(f"\n인덱싱 후 유저 수: {n_users}, 아이템 수: {n_items}")

# 유저별 상호작용 수
user_interaction_count = df.groupby("user_idx").size()
print("\n유저별 상호작용 수 통계:")
print(user_interaction_count.describe())

# ============================================================
# 추천 개수 K 규칙 (과제 제약조건 반영)
#  - 유저가 아이템을 n개 구매했다면 (여기서는 단순히 상호작용 수)
#    n <= 10 : 추천 개수 K = 2  (고정)
#    n > 10  : 추천 개수 K = floor(n * 0.2)
#  - 추천은 '이미 거래한 아이템' 이외에서 선택
# ============================================================

MAX_K = 100  # 추천 개수 상한 (너무 많은 추천 방지)

interaction_count_idx = df.groupby("user_idx").size().to_dict()

def get_k_for_user(count: int) -> int:
    if count <= 10:
        return 2
    k = int(count * 0.2)
    if k < 2:
        k = 2
    if k > MAX_K:
        k = MAX_K
    return k

user_k = {u_idx: get_k_for_user(c) for u_idx, c in interaction_count_idx.items()}

print("\n예시 K (추천 개수) 몇 명만 확인:")
for u_idx in list(user_k.keys())[:5]:
    print(f"  user {idx2user[u_idx]} (interaction={interaction_count_idx[u_idx]}): K={user_k[u_idx]}")


In [ ]:
# ============================================================
# 3. Train / Val / Test 분할 (user-wise)
# ============================================================

train_rows = []
val_rows = []
test_rows = []

for u_idx, u_df in df.groupby("user_idx"):
    # 유저별 셔플
    u_df = u_df.sample(frac=1.0, random_state=SEED)
    n = len(u_df)
    if n >= 3:
        n_train = int(n * 0.7)
        n_val = int(n * 0.15)
        if n_train < 1:
            n_train = 1
        if n_val < 1:
            n_val = 1
        if n_train + n_val >= n:
            n_train = n - 2
            n_val = 1
        train_rows.append(u_df.iloc[:n_train])
        val_rows.append(u_df.iloc[n_train:n_train + n_val])
        test_rows.append(u_df.iloc[n_train + n_val:])
    elif n == 2:
        train_rows.append(u_df.iloc[:1])
        val_rows.append(u_df.iloc[1:])
    else:  # n == 1
        train_rows.append(u_df)

train_df = pd.concat(train_rows, ignore_index=True)
val_df = pd.concat(val_rows, ignore_index=True) if val_rows else pd.DataFrame(columns=df.columns)
test_df = pd.concat(test_rows, ignore_index=True) if test_rows else pd.DataFrame(columns=df.columns)

print("데이터 분할 결과:")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# 유저별 train에서 이미 '본' 아이템 집합 (재추천 방지용)
user_train_items = defaultdict(set)
for u, i in zip(train_df["user_idx"].values, train_df["item_idx"].values):
    user_train_items[int(u)].add(int(i))

# 전체 (user,item) 쌍 집합 (negative sampling / 평가 시 사용)
all_edges = set(zip(df["user_idx"].values, df["item_idx"].values))

# ============================================================
# 4. Graph 구성 (LightGCN용)
# ============================================================

def build_graph_from_train(train_df, n_users, n_items, device):
    """user-item bipartite graph를 LightGCN용 edge_index, edge_weight로 변환"""
    users = train_df["user_idx"].values
    items = train_df["item_idx"].values

    # user -> item, item -> user 양방향 edge 구성
    edge_u2i = np.stack([users, items + n_users], axis=0)
    edge_i2u = np.stack([items + n_users, users], axis=0)
    edge_index = np.concatenate([edge_u2i, edge_i2u], axis=1)  # (2, 2E)

    edge_index = torch.LongTensor(edge_index)
    num_nodes = n_users + n_items

    deg = torch.zeros(num_nodes, dtype=torch.float32)
    deg = deg.scatter_add(0, edge_index[0], torch.ones(edge_index.size(1), dtype=torch.float32))

    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0.0

    edge_weight = deg_inv_sqrt[edge_index[0]] * deg_inv_sqrt[edge_index[1]]

    return edge_index.to(device), edge_weight.to(device)

edge_index, edge_weight = build_graph_from_train(train_df, n_users, n_items, device)
print("\n그래프 구성 완료 (edge 수: {})".format(edge_index.size(1)))


In [ ]:
# ============================================================
# 5. LightGCN 모델 정의
# ============================================================

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, n_layers=2):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.emb_dim = emb_dim
        self.n_layers = n_layers

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def forward(self, edge_index, edge_weight):
        # all_emb: (n_users + n_items, emb_dim)
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]

        for _ in range(self.n_layers):
            row, col = edge_index
            messages = all_emb[col] * edge_weight.unsqueeze(1)
            all_emb = torch.zeros_like(all_emb).scatter_add_(
                0,
                row.unsqueeze(1).expand(-1, self.emb_dim),
                messages,
            )
            embs.append(all_emb)

        final_emb = torch.mean(torch.stack(embs, dim=0), dim=0)
        user_emb_final = final_emb[:self.n_users]
        item_emb_final = final_emb[self.n_users:]

        return user_emb_final, item_emb_final

    def get_scores(self, users_idx_tensor, items_idx_tensor, edge_index, edge_weight):
        u_emb, i_emb = self.forward(edge_index, edge_weight)
        scores = (u_emb[users_idx_tensor] * i_emb[items_idx_tensor]).sum(dim=1)
        return scores

# ============================================================
# 6. 학습/평가 헬퍼 (BPR, negative sampling, metric 등)
# ============================================================

# 학습 하이퍼파라미터
EMB_DIM = 32
N_LAYERS = 2
LR = 5e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 50
NUM_NEG = 4  # 각 positive당 negative 샘플 수

model = LightGCN(n_users, n_items, emb_dim=EMB_DIM, n_layers=N_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

train_user_tensor = torch.LongTensor(train_df["user_idx"].values).to(device)
train_item_tensor = torch.LongTensor(train_df["item_idx"].values).to(device)

def sample_negative_items(num_users, num_neg=NUM_NEG, device=device):
    """각 user에 대해 num_neg 개의 negative item index 샘플링 (단, train interaction 제외는 엄밀히 보지 않음)."""
    return torch.randint(0, n_items, (num_users, num_neg), device=device)

def bpr_loss(pos_scores, neg_scores):
    # pos_scores: (N,), neg_scores: (N, num_neg)
    diff = pos_scores.unsqueeze(1) - neg_scores  # (N, num_neg)
    loss = -torch.log(torch.sigmoid(diff) + 1e-8).mean()
    return loss

@torch.no_grad()
def build_eval_samples(eval_df, num_neg_per_pos=1, device=device):
    """eval_df의 positive edge들과 동일 수의 random negative edge를 생성."""
    pos_users = eval_df["user_idx"].values
    pos_items = eval_df["item_idx"].values

    neg_users = []
    neg_items = []

    for u, i in zip(pos_users, pos_items):
        # 같은 user u에 대해, 관측되지 않은 item을 하나 샘플링
        while True:
            j = np.random.randint(0, n_items)
            if (u, j) not in all_edges:
                neg_users.append(u)
                neg_items.append(j)
                break

    return (
        torch.LongTensor(pos_users).to(device),
        torch.LongTensor(pos_items).to(device),
        torch.LongTensor(neg_users).to(device),
        torch.LongTensor(neg_items).to(device),
    )

@torch.no_grad()
def compute_scores_for_eval(model, eval_df, edge_index, edge_weight, device=device):
    """eval_df에 대해 pos/neg score, label을 반환."""
    if len(eval_df) == 0:
        return None, None

    pos_u, pos_i, neg_u, neg_i = build_eval_samples(eval_df, device=device)

    u_emb, i_emb = model(edge_index, edge_weight)

    pos_scores = (u_emb[pos_u] * i_emb[pos_i]).sum(dim=1).cpu().numpy()
    neg_scores = (u_emb[neg_u] * i_emb[neg_i]).sum(dim=1).cpu().numpy()

    labels = np.concatenate([np.ones_like(pos_scores), np.zeros_like(neg_scores)])
    scores = np.concatenate([pos_scores, neg_scores])

    return scores, labels

def find_best_threshold(scores, labels, num_grid=100):
    """Val set에서 F1이 최대가 되는 threshold를 찾는다."""
    thr_list = np.linspace(scores.min(), scores.max(), num_grid)
    best_thr = thr_list[0]
    best_f1 = 0.0
    best_o_ratio = 0.0

    for thr in thr_list:
        preds = (scores >= thr).astype(int)
        o_ratio = preds.mean()
        f1 = f1_score(labels, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr
            best_o_ratio = o_ratio

    return best_thr, best_f1, best_o_ratio

def evaluate_with_threshold(scores, labels, thr):
    preds = (scores >= thr).astype(int)
    o_ratio = preds.mean()
    auc = roc_auc_score(labels, scores)
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {
        "AUC": auc,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "O_ratio": o_ratio,
    }


In [ ]:
# ============================================================
# 7. 학습 루프 (LightGCN + BPR, full-batch)
# ============================================================

history_loss = []

print("\n===== Training Start (LightGCN + BPR) =====")
for epoch in range(1, EPOCHS + 1):
    model.train()
    start_time = time.time()

    # 전체 train interaction에 대해 한 번에 BPR loss 계산 (full-batch)
    user_emb_final, item_emb_final = model(edge_index, edge_weight)

    u = user_emb_final[train_user_tensor]                 # (N, d)
    pos = item_emb_final[train_item_tensor]               # (N, d)
    pos_scores = (u * pos).sum(dim=1)                     # (N,)

    neg_items = sample_negative_items(len(train_user_tensor), NUM_NEG, device=device)  # (N, num_neg)
    neg_emb = item_emb_final[neg_items]                   # (N, num_neg, d)
    u_expand = u.unsqueeze(1).expand_as(neg_emb)          # (N, num_neg, d)
    neg_scores = (u_expand * neg_emb).sum(dim=2)          # (N, num_neg)

    loss = bpr_loss(pos_scores, neg_scores)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    avg_loss = loss.item()
    history_loss.append(avg_loss)
    elapsed = time.time() - start_time

    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        print(f"[Epoch {epoch:03d}] loss={avg_loss:.4f} (time={elapsed:.1f}s)")

        # Validation set 평가
        if len(val_df) > 0:
            model.eval()
            val_scores, val_labels = compute_scores_for_eval(model, val_df, edge_index, edge_weight, device=device)
            if val_scores is not None:
                thr_tmp, f1_tmp, o_ratio_tmp = find_best_threshold(val_scores, val_labels)
                metrics_tmp = evaluate_with_threshold(val_scores, val_labels, thr_tmp)
                print(
                    f"  Val: AUC={metrics_tmp['AUC']:.4f}, F1={metrics_tmp['F1']:.4f}, "
                    f"O_ratio={metrics_tmp['O_ratio']:.3f}, thr~{thr_tmp:.4f}"
                )

print("===== Training Finished =====")

# 손실 곡선 시각화 (보고서용)
plt.figure(figsize=(6,4))
plt.plot(history_loss)
plt.xlabel("Epoch")
plt.ylabel("BPR Loss")
plt.title("LightGCN Training Loss")
plt.grid(True, alpha=0.3)
plt.show()

# ============================================================
# 8. Threshold 튜닝 및 Test 평가
# ============================================================

model.eval()
val_scores, val_labels = compute_scores_for_eval(model, val_df, edge_index, edge_weight, device=device)
test_scores, test_labels = compute_scores_for_eval(model, test_df, edge_index, edge_weight, device=device)

if val_scores is not None:
    BEST_THR, BEST_F1, BEST_O = find_best_threshold(val_scores, val_labels)
    print(f"\nVal 기준 최적 threshold: {BEST_THR:.4f} (F1={BEST_F1:.4f}, O_ratio={BEST_O:.3f})")
else:
    BEST_THR = np.median(test_scores)
    print("\nVal set이 비어 있어 test median을 threshold로 사용:", BEST_THR)

if test_scores is not None:
    test_metrics = evaluate_with_threshold(test_scores, test_labels, BEST_THR)
    print("\n===== Test Metrics (sampling 기반) =====")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")
else:
    print("Test set이 비어 있습니다.")


In [ ]:
# ============================================================
# 9. Inference용 래퍼 (임의의 test 파일에 대해 O/X 출력)
# ============================================================

class LightGCNRecommender:
    def __init__(self, model, edge_index, edge_weight, user_k, user_train_items, device=device):
        self.model = model
        self.edge_index = edge_index
        self.edge_weight = edge_weight
        self.user_k = user_k
        self.user_train_items = user_train_items
        self.device = device

        self.model.eval()
        with torch.no_grad():
            self.user_emb_final, self.item_emb_final = self.model(self.edge_index, self.edge_weight)

    def score_user_items(self, user_idx, item_indices):
        """특정 user_idx 와 여러 item_indices 에 대한 점수 벡터."""
        u_vec = self.user_emb_final[user_idx]                 # (d,)
        i_vecs = self.item_emb_final[item_indices]            # (N, d)
        scores = (u_vec.unsqueeze(0) * i_vecs).sum(dim=1)     # (N,)
        return scores.cpu().numpy()

    def recommend_for_group(self, user_id, group_df, threshold):
        """단일 user에 대해 group_df (해당 유저의 candidate rows)를 받아 O/X 결정."""
        results = []

        # 학습에 없던 유저 → 전부 X
        if user_id not in user2idx:
            for _, row in group_df.iterrows():
                results.append({"user": row["user"], "item": row["item"], "recommend": "X"})
            return results

        u_idx = user2idx[user_id]
        K = self.user_k.get(u_idx, 2)
        MIN_K = 2

        candidate_item_indices = []
        valid_rows = []

        # 1) 학습에 없는 item 또는 이미 거래한 item은 강제로 X
        for _, row in group_df.iterrows():
            item_id = row["item"]
            if item_id not in item2idx:
                results.append({"user": row["user"], "item": row["item"], "recommend": "X"})
                continue

            i_idx = item2idx[item_id]

            # 이미 train에서 본 아이템은 재추천하지 않음 ("n개 이외에서" 제약)
            if i_idx in self.user_train_items.get(u_idx, set()):
                results.append({"user": row["user"], "item": row["item"], "recommend": "X"})
                continue

            candidate_item_indices.append(i_idx)
            valid_rows.append(row)

        if len(candidate_item_indices) == 0:
            # 추천할 수 있는 후보가 없음
            return results

        candidate_item_indices = torch.LongTensor(candidate_item_indices).to(self.device)
        scores = self.score_user_items(u_idx, candidate_item_indices)

        # threshold 기준으로 1차 필터링
        indices = np.arange(len(scores))
        above = indices[scores >= threshold]

        if len(above) < MIN_K:
            # threshold 이상이 너무 적으면 top-MIN_K로 fallback
            top_idx = np.argsort(scores)[-MIN_K:]
            selected = set(top_idx)
        elif len(above) > K:
            # threshold 이상인데 너무 많으면 그 중에서 상위 K개만
            filtered_scores = scores[above]
            top_local = np.argsort(filtered_scores)[-K:]
            selected = set(above[top_local])
        else:
            selected = set(above)

        # valid_rows 순서에 맞춰 O/X 부여
        for idx, row in enumerate(valid_rows):
            rec = "O" if idx in selected else "X"
            results.append({"user": row["user"], "item": row["item"], "recommend": rec})

        return results

    def predict(self, test_df, threshold):
        all_results = []
        for user_id, group in test_df.groupby("user"):
            group_results = self.recommend_for_group(user_id, group, threshold)
            all_results.extend(group_results)

        return pd.DataFrame(all_results)

def print_formatted_results(pred_df):
    total = len(pred_df)
    rec_cnt = (pred_df["recommend"] == "O").sum()
    not_rec = total - rec_cnt

    print("====================")
    print(f"{'user':<7}{'item':<7}{'recommend':<10}")
    for _, row in pred_df.iterrows():
        print(f"{str(row['user']):<7}{str(row['item']):<7}{row['recommend']:<10}")
    print("====================")
    print(f"Total recommends = {rec_cnt}/{total}")
    print(f"Not recommend   = {not_rec}/{total}")

# ============================================================
# 10. 예시: 임의의 test 파일에 대해 추천 수행
# ============================================================

recommender = LightGCNRecommender(model, edge_index, edge_weight, user_k, user_train_items, device=device)

# Colab 채점 시에는 아래 test_path만 교체해서 실행
# 예) test_path = "/content/otc_test1.csv"
test_path = None  # 실제 채점용 파일 경로로 변경

if test_path is not None:
    test_file_df = pd.read_csv(test_path)
    # rating 컬럼이 들어있더라도 무시
    if "rating" in test_file_df.columns:
        test_file_df = test_file_df[["user", "item"]]

    preds = recommender.predict(test_file_df, BEST_THR)
    print_formatted_results(preds)
else:
    # 간단한 더미 예시 (형식 확인용)
    dummy_test = pd.DataFrame({
        "user": [1, 1, 4, 4],
        "item": [2, 10, 3, 20],
    })
    preds = recommender.predict(dummy_test, BEST_THR)
    print("샘플 dummy_test 결과:")
    print_formatted_results(preds)
